# 🧪 معمل التجارب: بناء مسار RAG باستخدام LCEL ومشغل الربط (|)
دليل عملي مباشر لتطبيق المحاضرة 28 مع شرح آلية عمل كل خطوة.

### 1. استيراد المكتبات وتحديد مسارات البيانات
**آلية العمل:**
نقوم بتحميل متغيرات البيئة من ملف `.env` (مثل مفتاح Groq)، واستيراد مكونات LangChain الأساسية لـ LCEL.
كما نحدد مسار مجلد `data` المركزي لحفظ المستندات وقاعدة البيانات لضمان بقاء مجلدات المشروع منظمة ونظيفة.

In [ ]:
# import standard libraries and load environment variables
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# resolve centralized project data paths
data_dir = Path("../../data") if Path("../../data").exists() else Path("./data")
docs_dir = str(data_dir / "chroma_sample_data")
db_dir = str(data_dir / "databases" / "chroma_vector_store")

# import langchain components
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

print("Libraries imported successfully!")

### 2. قراءة المستندات وتجهيز قاعدة المتجهات والمسترجع
**آلية العمل:**
1. يقرأ `DirectoryLoader` كافة الملفات النصية من المجلد.
2. يقطع `RecursiveCharacterTextSplitter` النصوص إلى قطع صغيرة (Chunks) متداخلة للحفاظ على ترابط المعنى.
3. يحول `HuggingFaceEmbeddings` النصوص إلى متجهات عددية دلالية.
4. تخزن المتجهات في `Chroma` داخل مسار `data/databases`.
5. يحول الفهرس إلى `Retriever` يبحث عن أفضل 3 نتائج متطابقة في المعنى دلالياً.

In [ ]:
# 1. load raw documents from data directory
loader = DirectoryLoader(docs_dir, glob="*.txt", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
raw_docs = loader.load()

# 2. split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunked_docs = text_splitter.split_documents(raw_docs)

# 3. initialize local embedding model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 4. load or create chroma vector store
vector_store = Chroma(
    persist_directory=db_dir,
    embedding_function=embeddings,
    collection_name="rag_collection"
)
if vector_store._collection.count() == 0:
    vector_store = Chroma.from_documents(
        documents=chunked_docs,
        embedding=embeddings,
        persist_directory=db_dir,
        collection_name="rag_collection"
    )

# 5. create retriever with top 3 matches
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
print(f"Retriever is ready with {vector_store._collection.count()} vectors in database!")

### 3. دالة دمج المستندات (`format_docs`)
**آلية العمل:**
المسترجع يُرجع قائمة من كائنات `Document`. هذه الدالة تأخذ محتوى كل مستند (`page_content`) وتدمجها في نص واحد مفصول بسطرين (`\n\n`) ليكون جاهزاً للحقن في خانة `{context}` داخل الـ Prompt.
وهي البديل الشفاف والمباشر لما كانت تفعله دالة `create_stuff_documents_chain` في السابق.

In [ ]:
# helper function to combine retrieved document chunks into a single string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# quick test of format_docs on sample retriever output
test_docs = retriever.invoke("What is deep learning?")
print("Sample formatted context:\n", format_docs(test_docs)[:140], "...")

### 4. إعداد قالب التوجيه (Prompt) ونموذج اللغة (Groq LLM)
**آلية العمل:**
- نحدد في الـ Prompt تعليمات صارمة للموديل بالاعتماد فقط على السياق المتاح (`context`) والإجابة بـ "I don't know" إذا لم تتوفر المعلومة لمنع الهلوسة.
- نربط نموذج `ChatGroq` فائق السرعة بدرجة حرارة منخفضة (`temperature=0.2`) لضمان دقة الإجابة والالتزام بالسياق.

In [ ]:
# prompt template enforcing strict context grounding
template = """You are a helpful AI assistant.
Answer the question using ONLY the provided context.
If you don't know the answer, say "I don't know."
Keep the answer concise.

Context:
{context}

Question:
{question}

Answer:"""
prompt = ChatPromptTemplate.from_template(template)

# initialize Groq chat model
llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_tokens=512
)
print("Prompt and LLM initialized successfully!")

### 5. بناء سلسلة الـ RAG الكاملة بمشغل الربط (`|`)
**آلية العمل:**
1. `retriever | format_docs`: يأخذ السؤال، يبحث في ChromaDB، ثم يمرر القطع لـ `format_docs` لتوضع في `{context}`.
2. `RunnablePassthrough()`: يمرر نفس السؤال النصي كما هو دون تغيير ليوضع في `{question}`.
3. الطرفان يعملان بالتوازي (Parallel Execution) داخل القاموس.
4. يمر القاموس الناتج إلى `prompt` لدمج النصوص.
5. يستلم `llm` الرسالة ويولد الرد ككائن `AIMessage`.
6. يستخرج `StrOutputParser()` النص الصافي فقط (`str`).

In [ ]:
# build full lcel rag chain using pipe operator
rag_chain_lcel = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
print("LCEL RAG chain created successfully!")

### 6. تجربة الاستعلام العادي والبث المباشر (Streaming)
**آلية العمل:**
- بفضل `RunnablePassthrough`، يمكننا مناداة السلسلة بنص مباشر (`str`) عبر `.invoke(query)` دون الحاجة لقواميس معقدة.
- بفضل طبيعة LCEL، تدعم السلسلة ميزة البث اللحظي الحرفي `.stream(query)` فورياً دون أي إعدادات إضافية.

In [ ]:
# test 1: invoke pipeline with a direct string query
query = "What is deep learning and how does it differ from traditional machine learning?"
answer = rag_chain_lcel.invoke(query)
print(f"Question: {query}\n")
print(f"Answer:\n{answer}\n")

print("=" * 60)
# test 2: real-time token-by-token streaming
stream_query = "What are CNNs best used for?"
print(f"Streaming answer for: '{stream_query}'\n")

for chunk in rag_chain_lcel.stream(stream_query):
    print(chunk, end="", flush=True)
print()